# Vector vs Raster: Core Data Models

Companion notebook for the **EcoGeo Tutor** tutorial: *Vector vs Raster: Core Data Models*.

Loads, inspects, visualizes, and combines both spatial data models using `geopandas`
and `rasterio`, finishing with zonal statistics (mean elevation per river basin).

**Data you'll need:**
- A river basin polygon shapefile — e.g. from [Natural Earth](https://www.naturalearthdata.com/downloads/10m-physical-vectors/)
- A DEM (Digital Elevation Model) GeoTIFF — e.g. SRTM 30m from [USGS EarthExplorer](https://earthexplorer.usgs.gov)

Update the file paths below to match your own downloads.


## Setup

In [ ]:
!pip install geopandas rasterio rasterstats matplotlib -q


## 1. Vector — load a river basin polygon

In [ ]:
import geopandas as gpd
import rasterio
import rasterio.plot
import matplotlib.pyplot as plt

basins = gpd.read_file("river_basins.shp")

print(f"CRS: {basins.crs}")
print(f"Geometry type: {basins.geometry.geom_type.unique()}")
print(f"Number of features: {len(basins)}")

# Project to metres, then calculate area in km²
basins = basins.to_crs(epsg=3857)
basins["area_km2"] = basins.geometry.area / 1e6
print(basins[["name", "area_km2"]].head())

basins.plot(column="area_km2", cmap="Blues", legend=True, figsize=(10, 6))
plt.title("River Basins - coloured by area (km2)")
plt.axis("off")
plt.tight_layout()
plt.savefig("vector_basins.png", dpi=150)
plt.show()


## 2. Raster — load a DEM

In [ ]:
with rasterio.open("dem_30m.tif") as src:
    dem = src.read(1)
    profile = src.profile
    bounds = src.bounds

print(f"Raster shape: {dem.shape}")
print(f"Resolution: {profile['transform'][0]:.1f} m")
print(f"NoData value: {profile['nodata']}")
print(f"Elevation range: {dem.min():.0f} - {dem.max():.0f} m")

fig, ax = plt.subplots(figsize=(10, 6))
with rasterio.open("dem_30m.tif") as src:
    rasterio.plot.show(src, ax=ax, cmap="terrain", title="Digital Elevation Model (30m)")
plt.tight_layout()
plt.savefig("raster_dem.png", dpi=150)
plt.show()


## 3. Combining both — zonal statistics

What is the mean elevation inside each river basin?

In [ ]:
from rasterstats import zonal_stats
import numpy as np

with rasterio.open("dem_30m.tif") as src:
    stats = zonal_stats(
        basins.to_crs(src.crs),
        "dem_30m.tif",
        stats=["mean", "min", "max", "std"],
        nodata=src.nodata
    )

basins["elev_mean"] = [s["mean"] for s in stats]
basins["elev_max"]  = [s["max"]  for s in stats]

print(basins[["name", "area_km2", "elev_mean", "elev_max"]].head(10))


## Common mistakes to avoid

- **CRS mismatch** — always check vector and raster share the same CRS before combining. Reproject with `.to_crs()`.
- **Forgetting NoData** — raster files often contain a NoData value (e.g. -9999). Always pass `nodata=` when reading, or statistics will be wrong.
- **Wrong geometry type** — a GeoDataFrame can contain mixed geometry types; check `.geom_type.unique()` first.
- **Resolution confusion** — a 10m raster and a 30m raster contain very different detail. Print `profile['transform'][0]` before analysis.

---
*Companion notebook for the EcoGeo Tutor tutorial on rcafe.vercel.app*
